## Описание

Ноутбук забирает товары из DummyJSON и пишет их в таблицу `stg.products`.

Источник: `GET {API_BASE_URL}/products`, страницы через `limit` и `skip`.
Цель: Postgres, таблица `stg.products`. Её создаёт `init/init.sql`.
Повторный запуск не добавляет дубли: `ON CONFLICT (product_id) DO NOTHING`.


## Импорты


In [ ]:
import logging
import os
from pathlib import Path

import pandas as pd
import psycopg2
import requests
from dotenv import load_dotenv
from psycopg2.extras import execute_values


## Конфигурация

Пароль и адрес API берутся из `.env`. Файл ищется в текущей папке и на уровень выше.


In [ ]:
def find_env_file():
    for path in (Path.cwd() / ".env", Path.cwd().parent / ".env"):
        if path.is_file():
            return path
    raise FileNotFoundError(
        "Не найден .env. Скопируй .env.example в .env и запусти ноутбук из корня проекта или из notebooks/."
    )


env_path = find_env_file()
load_dotenv(env_path)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s - %(message)s",
    force=True,
)
logger = logging.getLogger("etl")

needed = [
    "POSTGRES_HOST",
    "POSTGRES_PORT",
    "POSTGRES_DB",
    "POSTGRES_USER",
    "POSTGRES_PASSWORD",
    "API_BASE_URL",
    "API_PAGE_SIZE",
]
missing = [name for name in needed if not os.getenv(name)]
if missing:
    raise RuntimeError("В .env не хватает переменных: " + ", ".join(missing))

pg = {
    "host": os.environ["POSTGRES_HOST"],
    "port": int(os.environ["POSTGRES_PORT"]),
    "dbname": os.environ["POSTGRES_DB"],
    "user": os.environ["POSTGRES_USER"],
    "password": os.environ["POSTGRES_PASSWORD"],
}
api_base_url = os.environ["API_BASE_URL"].rstrip("/")
page_size = int(os.environ["API_PAGE_SIZE"])
max_pages = 50

logger.info(
    "Конфиг из %s. API=%s, page_size=%s, postgres=%s:%s/%s",
    env_path,
    api_base_url,
    page_size,
    pg["host"],
    pg["port"],
    pg["dbname"],
)


## Extract

Читаю страницы, пока не наберу `total` товаров из ответа.


In [ ]:
def fetch_products():
    products = []
    skip = 0
    total = None

    for page in range(1, max_pages + 1):
        url = api_base_url + "/products"
        params = {"limit": page_size, "skip": skip}
        try:
            response = requests.get(url, params=params, timeout=30)
            response.raise_for_status()
            payload = response.json()
        except requests.Timeout:
            logger.error("Таймаут, страница %s, skip=%s", page, skip)
            raise RuntimeError("API не ответил за 30 секунд: " + url)
        except requests.RequestException as exc:
            logger.error("Ошибка запроса %s: %s", url, exc)
            raise RuntimeError("Не удалось получить данные из API: %s" % exc)

        if "products" not in payload or "total" not in payload:
            raise RuntimeError("Странный ответ API, ключи: %s" % list(payload))

        batch = payload["products"]
        total = int(payload["total"])
        products.extend(batch)
        logger.info(
            "страница %s, skip=%s, получено %s, всего %s",
            page,
            skip,
            len(batch),
            total,
        )
        if not batch or len(products) >= total:
            break
        skip += page_size

    if not products:
        raise RuntimeError("API вернул пустой каталог, загружать нечего.")
    if total is not None and len(products) < total:
        raise RuntimeError("Остановился после %s страниц, проверь page_size." % max_pages)

    logger.info("Extract: %s записей", len(products))
    return products


raw_products = fetch_products()


## Transform

Оставляю нужные поля. Картинки, отзывы, размеры и `meta` выкидываю. Типы привожу к числам, пустые строки и дубли по `product_id` убираю.


In [ ]:
columns = [
    "id",
    "title",
    "description",
    "category",
    "price",
    "discountPercentage",
    "rating",
    "stock",
    "brand",
    "sku",
]

raw_df = pd.DataFrame(raw_products)
missing_cols = [col for col in columns if col not in raw_df.columns]
if missing_cols:
    raise RuntimeError("В ответе API нет колонок: " + ", ".join(missing_cols))

df = raw_df[columns].rename(columns={
    "id": "product_id",
    "discountPercentage": "discount_percentage",
})

before = len(df)
for col in ("product_id", "price", "discount_percentage", "rating", "stock"):
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["title"] = df["title"].astype("string").str.strip()
df["description"] = df["description"].astype("string")
df["category"] = df["category"].astype("string").str.strip()
df["brand"] = df["brand"].astype("string")
df["sku"] = df["sku"].astype("string")

df = df.dropna(subset=["product_id", "title", "category", "price", "stock"])
df = df[(df["title"] != "") & (df["category"] != "")]
df = df[df["price"] >= 0]
df = df.drop_duplicates(subset=["product_id"])
df["product_id"] = df["product_id"].astype(int)
df["stock"] = df["stock"].astype(int)
df = df.sort_values("product_id").reset_index(drop=True)

logger.info("Transform: было %s, осталось %s, убрал %s", before, len(df), before - len(df))
if df.empty:
    raise RuntimeError("После очистки не осталось строк для загрузки.")

df.head()


## Load

Пишу строки в `stg.products`. Если `product_id` уже есть, строка пропускается.


In [ ]:
rows = []
for row in df.itertuples(index=False):
    rows.append((
        int(row.product_id),
        str(row.title),
        None if pd.isna(row.description) else str(row.description),
        str(row.category),
        float(row.price),
        None if pd.isna(row.discount_percentage) else float(row.discount_percentage),
        None if pd.isna(row.rating) else float(row.rating),
        int(row.stock),
        None if pd.isna(row.brand) else str(row.brand),
        None if pd.isna(row.sku) else str(row.sku),
    ))

insert_sql = """
    INSERT INTO stg.products (
        product_id, title, description, category, price,
        discount_percentage, rating, stock, brand, sku
    )
    VALUES %s
    ON CONFLICT (product_id) DO NOTHING
"""

conn = None
try:
    try:
        conn = psycopg2.connect(**pg)
    except psycopg2.OperationalError as exc:
        logger.error("Не удалось подключиться к Postgres: %s", exc)
        raise RuntimeError("Postgres недоступен. Проверь контейнер: docker compose ps")

    try:
        with conn:
            with conn.cursor() as cur:
                execute_values(cur, insert_sql, rows, page_size=100)
                inserted = cur.rowcount
    except psycopg2.Error as exc:
        logger.error("Ошибка записи в Postgres: %s", exc)
        raise RuntimeError("Загрузка в stg.products не удалась: %s" % exc)
finally:
    if conn is not None:
        conn.close()

logger.info(
    "Load: отправлено %s, вставлено %s, пропущено %s",
    len(rows),
    inserted,
    len(rows) - inserted,
)


## Проверка

`SELECT COUNT(*)` и первые 5 строк.


In [ ]:
conn = None
try:
    try:
        conn = psycopg2.connect(**pg)
    except psycopg2.OperationalError as exc:
        logger.error("Не удалось подключиться к Postgres: %s", exc)
        raise RuntimeError("Postgres недоступен, проверка не выполнена.")

    try:
        with conn:
            with conn.cursor() as cur:
                cur.execute("SELECT COUNT(*) FROM stg.products")
                total_rows = cur.fetchone()[0]
                cur.execute(
                    """
                    SELECT product_id, title, category, price, discount_percentage,
                           rating, stock, brand, sku
                    FROM stg.products
                    ORDER BY product_id
                    LIMIT 5
                    """
                )
                preview_rows = cur.fetchall()
                columns = [desc[0] for desc in cur.description]
    except psycopg2.Error as exc:
        logger.error("Проверочный запрос не выполнился: %s", exc)
        raise RuntimeError("Не удалось прочитать stg.products: %s" % exc)
finally:
    if conn is not None:
        conn.close()

logger.info("В stg.products %s строк", total_rows)
preview = pd.DataFrame(preview_rows, columns=columns)
print("COUNT(*) =", total_rows)
preview
